 Import Libraries

In [13]:
import pandas as pd
import numpy as np
import re
import string

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# ML
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\srini\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\srini\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

Load Dataset

In [14]:
df = pd.read_csv("C:/Users/srini/Downloads/cleaned_reviews.csv/cleaned_reviews.csv")  
print(df.head())
print(df.columns)
print(df['sentiments'].value_counts())


  sentiments                                     cleaned_review  \
0   positive  i wish would have gotten one earlier love it a...   
1    neutral  i ve learned this lesson again open the packag...   
2    neutral          it is so slow and lags find better option   
3    neutral  roller ball stopped working within months of m...   
4    neutral  i like the color and size but it few days out ...   

   cleaned_review_length  review_score  
0                     19           5.0  
1                     88           1.0  
2                      9           2.0  
3                     12           1.0  
4                     21           1.0  
Index(['sentiments', 'cleaned_review', 'cleaned_review_length',
       'review_score'],
      dtype='object')
sentiments
positive    9503
neutral     6303
negative    1534
Name: count, dtype: int64


 NLP Preprocessing Function

In [28]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)  # remove URLs
    text = re.sub(r"[^a-zA-Z]", " ", text)  # remove special chars
    words = text.split()
    
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    
    return " ".join(words)

df['processed_text'] = df['cleaned_review'].astype(str).apply(preprocess_text)

 Feature Engineering

In [29]:
#Bag of Words
bow = CountVectorizer(max_features=5000)
X_bow = bow.fit_transform(df['processed_text'])

In [30]:
#TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df['processed_text'])

Define Target

In [31]:
y = df['sentiments']

Train-Test Split

In [32]:
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

Model Training

In [33]:
#Logistic Regression
lr = LogisticRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [22]:
#Naive Bayes
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

In [23]:
#Decision Tree
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)
y_pred_dt = dt.predict(X_test)

Evaluation Function

In [34]:
def evaluate_model(y_test, y_pred, model_name):
    print(f"\n🔹 {model_name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))

Compare Models

In [35]:
evaluate_model(y_test, y_pred_lr, "Logistic Regression")
evaluate_model(y_test, y_pred_nb, "Naive Bayes")
evaluate_model(y_test, y_pred_dt, "Decision Tree")


🔹 Logistic Regression
Accuracy: 0.8220876585928489
Precision: 0.8214060364630282
Recall: 0.8220876585928489
F1 Score: 0.8144554504984135

Classification Report:
               precision    recall  f1-score   support

    negative       0.75      0.37      0.49       336
     neutral       0.74      0.83      0.78      1253
    positive       0.89      0.90      0.89      1879

    accuracy                           0.82      3468
   macro avg       0.79      0.70      0.72      3468
weighted avg       0.82      0.82      0.81      3468


🔹 Naive Bayes
Accuracy: 0.6825259515570934
Precision: 0.7030341544182498
Recall: 0.6825259515570934
F1 Score: 0.638307992408406

Classification Report:
               precision    recall  f1-score   support

    negative       1.00      0.01      0.02       336
     neutral       0.60      0.51      0.55      1253
    positive       0.72      0.92      0.81      1879

    accuracy                           0.68      3468
   macro avg       0.77      0

In [36]:
"""
Conclusion:
- TF-IDF performed better than Bag of Words.
- Logistic Regression gave highest accuracy (usually).
- Naive Bayes is faster but slightly less accurate.
- Decision Tree may overfit.

Best Model: Logistic Regression
Best Feature: TF-IDF
"""

'\nConclusion:\n- TF-IDF performed better than Bag of Words.\n- Logistic Regression gave highest accuracy (usually).\n- Naive Bayes is faster but slightly less accurate.\n- Decision Tree may overfit.\n\nBest Model: Logistic Regression\nBest Feature: TF-IDF\n'